In [158]:
import pickle
from pathlib import Path
from huggingface_hub import snapshot_download
import pandas as pd
import math


## Lecturas Elementos Externos

### Leer dataset de errores

In [159]:
data_dir = snapshot_download(
    repo_id="jhonrayo99/nlp-tarea-2-ngramas",
    repo_type="dataset",
)
archivo_errores = Path(data_dir) / "spelling_errors.tsv"
errores = pd.read_csv(
    archivo_errores,
    sep="\t"
)

errores.head()
print(errores.columns)
errores.info()

Fetching 60 files: 100%|██████████| 60/60 [00:00<00:00, 2168.68it/s]

Index(['author', 'context', 'error', 'correction', 'operation'], dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   author      60 non-null     str  
 1   context     60 non-null     str  
 2   error       60 non-null     str  
 3   correction  60 non-null     str  
 4   operation   60 non-null     str  
dtypes: str(5)
memory usage: 9.4 KB


In [160]:
errores.head(10)

,author,context,error,correction,operation
0,austen,but one neer does form a just idea of any body...,neer,never,deletion
1,austen,"`` and do you see her , sir , tolezably often ...",tolezably,tolerably,substitution
2,austen,"crxied harriet , colouring , and astonished .",crxied,cried,insertion
3,austen,`` yes ; jane says she is sure they will ; but...,sitution,situation,deletion
4,austen,"harriet submitted , thogh her mind could hardl...",thogh,though,deletion
5,austen,"thzir propriety , simplicity , and elegance , ...",thzir,their,substitution
6,austen,how cozld she have exposed herself to such ill...,cozld,could,substitution
7,austen,"she owned that , considering evzry thing , she...",evzry,every,substitution
8,austen,"it was shrt , but expressed good sense , warm ...",shrt,short,deletion
9,austen,"he will thxink all the happiness , all the adv...",thxink,think,insertion


### Leer Modelos de bigramas

In [161]:
with open("modelos_bigramas.pkl", "rb") as archivo:
    recursos_corrector = pickle.load(archivo)

modelo_austen = recursos_corrector["austen"]
modelo_twain = recursos_corrector["twain"]

vocabulario_austen = modelo_austen["vocabulario"]
vocabulario_twain = modelo_twain["vocabulario"]

## Corrección ortográfica

### Calcular distancias de edicion

In [162]:
def distancia_edicion(x, w):
    """
    Calcula la distancia minima de edicion entre dos strings utilizando
    unicamente insercion, borrado y sustitucion de caracteres.

    Parametros
    ----------
    x : str
        Primera cadena.

    w : str
        Segunda cadena.

    Retorna
    -------
    int
        Numero minimo de operaciones necesarias para transformar
        x en w.
    """

    m = len(x)
    n = len(w)

   # Crear matriz
    matriz = [[0] * (n + 1) for _ in range(m + 1)]

    # Caso base:
    # transformar x[:i] en cadena vacia requiere i borrados
    for i in range(m + 1):
        matriz[i][0] = i

    # transformar cadena vacia en w[:j] requiere j inserciones
    for j in range(n + 1):
        matriz[0][j] = j

    # Llenar la matriz
    for i in range(1, m + 1):
        for j in range(1, n + 1):

            # Si los caracteres son iguales, no hay costo
            if x[i - 1] == w[j - 1]:
                costo = 0
            else:
                costo = 1

            insercion = matriz[i][j - 1] + 1
            borrado = matriz[i - 1][j] + 1
            sustitucion = matriz[i - 1][j - 1] + costo

            matriz[i][j] = min(
                insercion,
                borrado,
                sustitucion
            )

    return matriz[m][n]

### Generar Candidatos

In [163]:
def generar_candidatos(x, vocabulario, max_distancia=2):
    """
    Genera un conjunto de palabras candidatas para corregir un token erroneo.

    La funcion recorre todas las palabras del vocabulario y selecciona aquellas
    cuya distancia de edicion respecto al token erroneo x sea menor o igual a
    max_distancia. Los tokens especiales <s> y </s> son excluidos del conjunto
    de candidatos, ya que representan marcadores de inicio y fin de oracion y
    no corresponden a posibles correcciones.

    Parametros:
        x: Token erroneo que se desea corregir.
        vocabulario: Conjunto de palabras disponibles como posibles candidatos.
        max_distancia: Distancia maxima de edicion permitida. Por defecto es 2.

    Retorna:
        Una lista con las palabras del vocabulario que pueden ser candidatas
        para corregir el token x.
    """
    
    candidatos = []
    tokens_especiales = {"<s>", "</s>"}
    
    for w in vocabulario:
        
        if w in tokens_especiales:
            continue
            
        distancia = distancia_edicion(x, w)
        
        if distancia <= max_distancia:
            candidatos.append(w)
    
    return candidatos

In [164]:
def reemplazar_error(contexto, error, candidato):
    """
    Reemplaza la primera aparicion de un token erroneo por una palabra candidata.

    El contexto se divide en tokens utilizando los espacios como separadores.
    Luego se busca la primera aparicion del error y se reemplaza por el
    candidato seleccionado. Solo se modifica una ocurrencia del error.

    Parametros:
        contexto: Oracion original que contiene el token erroneo.
        error: Token que se desea reemplazar.
        candidato: Palabra que reemplazara el error.

    Retorna:
        Una lista de tokens correspondiente a la oracion con el error
        reemplazado por el candidato.
    """
    tokens = contexto.split()
    reemplazado = False
    
    for i, token in enumerate(tokens):
        
        if token == error and not reemplazado:
            tokens[i] = candidato
            reemplazado = True
    
    return tokens

In [165]:
palabra_error = "teh"

candidatos = generar_candidatos(
    palabra_error,
    vocabulario_austen
)

print(f"Palabra erronea: {palabra_error}")
print(f"Numero de candidatos: {len(candidatos)}")
print(candidatos[:20])

Palabra erronea: teh
Numero de candidatos: 91
[np.str_('wen'), np.str_('pey'), np.str_('fed'), np.str_('tax'), np.str_('ash'), np.str_('wet'), np.str_('beg'), np.str_('key'), np.str_('sah'), np.str_('teeth'), np.str_('term'), np.str_('se'), np.str_('tin'), np.str_('try'), np.str_('the'), np.str_('red'), np.str_('step'), np.str_('tell'), np.str_('text'), np.str_('lee')]


### Seleccionar la mejor oracion

In [166]:
def log_probabilidad_oracion(oracion, modelo, vocabulario):
    """
    Calcula la probabilidad(se utiliza logaritmos por facilidad de calculo)
    de una oracion utilizando un modelo de bigramas con suavizado de Laplace.

    Se agregan los marcadores especiales de inicio <s> y fin </s> para evaluar
    tambien las probabilidades del primer y ultimo token de la oracion. La
    probabilidad de cada bigrama se calcula utilizando sus conteos almacenados
    en el modelo y se suman sus logaritmos para obtener la puntuacion total de
    la oracion.

    Parametros:
        oracion: Lista de tokens que representa la oracion a evaluar.
        modelo: Diccionario que contiene los conteos de bigramas y contextos.
        vocabulario: Conjunto de palabras utilizado para calcular el tamaño
                     del vocabulario en el suavizado de Laplace.

    Retorna:
        La suma de las log-probabilidades de todos los bigramas de la oracion.
    """

    log_probabilidad = 0.0
    
    # Agregamos los marcadores de inicio y fin
    tokens = ["<s>"] + oracion + ["</s>"]
    
    for i in range(1, len(tokens)):
        
        bigrama = (tokens[i - 1], tokens[i])
        contexto = (tokens[i - 1],)
        
        numerador = modelo["counts"][bigrama] + 1
        denominador = (modelo["context_counts"][contexto]+ len(vocabulario) )
        
        probabilidad = numerador / denominador
        
        log_probabilidad += math.log(probabilidad)
    
    return log_probabilidad

In [167]:
def seleccionar_candidato_lm(contexto,error,candidatos,modelo,vocabulario):
    """
    Selecciona el mejor candidato para corregir un error utilizando
    unicamente el modelo de lenguaje.

    Para cada palabra candidata, se reemplaza el error dentro del contexto y
    se calcula la log-probabilidad de la oracion resultante. Se selecciona el
    candidato cuya oracion obtiene la mayor log-probabilidad segun el modelo
    de bigramas.

    Parametros:
        contexto: Oracion original que contiene el token erroneo.
        error: Token que se desea corregir.
        candidatos: Lista de posibles palabras candidatas.
        modelo: Modelo de bigramas utilizado para calcular probabilidades.
        vocabulario: Conjunto de palabras del modelo.

    Retorna:
        Una tupla con el candidato seleccionado y su correspondiente
        log-probabilidad.
    """
    
    mejor_candidato = None
    
    #Se inicializa en - infinito con eso grantizo que el primer candidato siempre sera mejor que el valor inicial, sin importar que tan negativo sea su score.
    mejor_log_probabilidad = float("-inf")
    
    for candidato in candidatos:
        
        # Crear la oracion reemplazando el error
        oracion = reemplazar_error(contexto,error,candidato)
        
        # Calcular su log-probabilidad
        log_probabilidad = log_probabilidad_oracion(oracion,modelo,vocabulario)
        
        # Actualizar si encontramos una mejor opcion
        if log_probabilidad > mejor_log_probabilidad:
            mejor_log_probabilidad = log_probabilidad
            mejor_candidato = candidato
    
    return mejor_candidato, mejor_log_probabilidad

In [168]:
def seleccionar_candidato_heuristico(contexto,error,candidatos,modelo,vocabulario):

    """
    Selecciona el mejor candidato para corregir un error utilizando una
    combinacion entre el modelo de lenguaje y la distancia de edicion.

    Para cada candidato, se reemplaza el error dentro del contexto y se calcula
    la log-probabilidad de la oracion resultante. Posteriormente, se aplica una
    penalizacion proporcional a la distancia de edicion entre el error y el
    candidato. Se selecciona el candidato con el mayor score final.

    Parametros:
        contexto: Oracion original que contiene el token erroneo.
        error: Token que se desea corregir.
        candidatos: Lista de posibles palabras candidatas.
        modelo: Modelo de bigramas utilizado para calcular probabilidades.
        vocabulario: Conjunto de palabras del modelo.

    Retorna:
        Una tupla con el candidato seleccionado y su score heuristico.
    """
    
    
    mejor_candidato = None
    #Se inicializa en - infinito con eso grantizo que el primer candidato siempre sera mejor que el valor inicial, sin importar que tan negativo sea su score.
    mejor_score = float("-inf") 
    
    for candidato in candidatos:
        
        # Crear la oracion con el candidato
        oracion = reemplazar_error(contexto,error,candidato)
        
        # Calcular log-probabilidad
        log_probabilidad = log_probabilidad_oracion(oracion,modelo,vocabulario)
        
        # Calcular distancia de edicion
        distancia = distancia_edicion(error,candidato)
        
        # Aplicar heuristica
        score = log_probabilidad - 1.5 * distancia
        
        # Conservar el mejor candidato
        if score > mejor_score:
            mejor_score = score
            mejor_candidato = candidato
    
    return mejor_candidato, mejor_score

### Evaluacion de los errores

In [169]:
recursos_por_autor = {
    "austen": {
        "modelo": modelo_austen,
        "vocabulario": vocabulario_austen
    },
    "twain": {
        "modelo": modelo_twain,
        "vocabulario": vocabulario_twain
    }
}

def corregir_error(fila):
    
    autor = fila["author"]
    contexto = fila["context"]
    error = fila["error"]
    
    # Obtener recursos del autor
    modelo = recursos_por_autor[autor]["modelo"]
    vocabulario = recursos_por_autor[autor]["vocabulario"]
    
    # Generar candidatos
    candidatos = generar_candidatos(error,vocabulario)
    
    # Metodo 1: modelo de lenguaje
    candidato_lm, log_probabilidad = seleccionar_candidato_lm(contexto=contexto,error=error,
                                                              candidatos=candidatos, modelo=modelo,vocabulario=vocabulario)
    
    # Metodo 2: heuristica
    candidato_heuristico, score_heuristico = seleccionar_candidato_heuristico(contexto=contexto,error=error,
                                                                              candidatos=candidatos,modelo=modelo,vocabulario=vocabulario)
    
    return {
        "candidato_lm": candidato_lm,
        "log_probabilidad": log_probabilidad,
        "candidato_heuristico": candidato_heuristico,
        "score_heuristico": score_heuristico,
        "num_candidatos": len(candidatos)
    }

In [170]:
# Ejecutar correccion sobre todo el dataset
resultados = []

for _, fila in errores.iterrows():
    
    resultado = corregir_error(fila)
    
    resultados.append({
        "author": fila["author"],
        "context": fila["context"],
        "error": fila["error"],
        "correccion": fila["correction"],
        "operacion": fila["operation"],
        "prediccion_lm": resultado["candidato_lm"],
        "prediccion_heuristica": resultado["candidato_heuristico"],
        "num_candidatos": resultado["num_candidatos"]
    })


# Convertir resultados a DataFrame
resultados_df = pd.DataFrame(resultados)


# Determinar si cada prediccion fue correcta
resultados_df["correcto_lm"] = (resultados_df["prediccion_lm"]== resultados_df["correccion"])
resultados_df["correcto_heuristica"] = (resultados_df["prediccion_heuristica"]== resultados_df["correccion"])


# porcentaje de correcciones correctas
p_correcciones_lm = resultados_df["correcto_lm"].mean() * 100
p_correcciones_heuristica = (resultados_df["correcto_heuristica"].mean() * 100)


print("EVALUACIÓN GENERAL")
print("-" * 40)
print(f"Accuracy LM: {p_correcciones_lm:.2f}%")
print(f"Accuracy Heuristica: {p_correcciones_heuristica:.2f}%")

EVALUACIÓN GENERAL
----------------------------------------
Accuracy LM: 76.67%
Accuracy Heuristica: 88.33%


In [171]:
def calcular_resumen(df, agrupacion, categoria):
    
    resumen = (df.groupby(agrupacion)[["correcto_lm", "correcto_heuristica"]].mean().mul(100).reset_index() )
    
    resumen = resumen.rename(columns={
        "correcto_lm": "Correcciones LM (%)",
        "correcto_heuristica": "Correcciones Heuristicas(%)"
    })
    
    resumen.insert(0, "Categoria", categoria)
    
    return resumen


# Resultado general
resumen_general = pd.DataFrame([{
    "Categoria": "General",
    "Autor": "Total",
    "Operacion": "Total",
    "Correcciones LM (%)": p_correcciones_lm,
    "Correcciones Heuristica (%)": p_correcciones_heuristica
}])


# Resultados por autor
resumen_autor = calcular_resumen(resultados_df,"author","Autor").rename(columns={"author": "Autor"})
resumen_autor["Operacion"] = "Total"

# Resultados por operacion
resumen_operacion = calcular_resumen(resultados_df,"operacion","Operacion").rename(columns={"operation": "Operacion"})
resumen_operacion["Autor"] = "Total"


# Resultados por autor y operacion
resumen_autor_operacion = calcular_resumen(resultados_df,["author", "operacion"],"Autor + Operacion").rename(columns={"author": "Autor","operacion": "Operacion"})


# Unir todos los resultados
resumen_completo = pd.concat([
    resumen_general,
    resumen_autor,
    resumen_operacion,
    resumen_autor_operacion
], ignore_index=True)


# Organizar y redondear
resumen_completo = resumen_completo[
    [
        "Categoria",
        "Autor",
        "Operacion",
        "Correcciones LM (%)",
        "Correcciones Heuristica (%)"
    ]
].round(2)


resumen_completo

,Categoria,Autor,Operacion,Correcciones LM (%),Correcciones Heuristica (%)
0,General,Total,Total,76.67,88.33
1,Autor,austen,Total,80.00,NaN
2,Autor,twain,Total,73.33,NaN
3,Operacion,Total,NaN,55.00,NaN
4,Operacion,Total,NaN,85.00,NaN
5,Operacion,Total,NaN,90.00,NaN
6,Autor + Operacion,austen,deletion,60.00,NaN
7,Autor + Operacion,austen,insertion,90.00,NaN
8,Autor + Operacion,austen,substitution,90.00,NaN
9,Autor + Operacion,twain,deletion,50.00,NaN


La tabla muestra que el metodo heuristico obtuvo mejores resultados que el modelo basado unicamente en lenguaje. En general, la precision aumento de 76.67% a 88.33%. La mejora fue especialmente grande para Twain, donde paso de 73.33% a 93.33%, mientras que para Austen el cambio fue menor, de 80% a 83.33%. Por tipo de edicion, la mayor mejora ocurrio en los errores de eliminacion, que pasaron de 55% a 75%, seguidos por insercion y sustitucion. Esto muestra que considerar la distancia entre el error y cada candidato ayuda al corrector a evitar palabras que pueden tener sentido en el contexto, pero que no son ortograficamente las mas cercanas al error original.

In [172]:
errores_corrector = resultados_df[
    (~resultados_df["correcto_lm"])
    | (~resultados_df["correcto_heuristica"])
]

errores_corrector[
    [
        "author",
        "context",
        "error",
        "correccion",
        "operacion",
        "prediccion_lm",
        "correcto_lm",
        "prediccion_heuristica",
        "correcto_heuristica"
    ]
]

,author,context,error,correccion,operacion,prediccion_lm,correcto_lm,prediccion_heuristica,correcto_heuristica
5,austen,"thzir propriety , simplicity , and elegance , ...",thzir,their,substitution,this,False,their,True
8,austen,"it was shrt , but expressed good sense , warm ...",shrt,short,deletion,sure,False,sure,False
10,austen,he is desperately in love and mens to marry he...,mens,means,deletion,me,False,me,False
19,austen,"`` come , come , '' cred emma , feeling this t...",cred,cried,deletion,fed,False,cured,False
20,austen,`` it is too smll — wants strength .,smll,small,deletion,well,False,well,False
28,austen,`` i know _that_ is the feexling of you all .,feexling,feeling,insertion,feelings,False,feelings,False
32,twain,no ; the raggedest tramp in the pit wold smile...,wold,would,deletion,will,False,would,True
40,twain,for surely ye mean that ye woxuld not betray m...,woxuld,would,insertion,could,False,would,True
41,twain,"`` now we are safe , '' i said , `` but it was...",clse,close,deletion,lie,False,close,True
43,twain,"the pillory -- '' '' oh , wait , broher ; say ...",broher,brother,deletion,broken,False,brother,True


La tabla muestra los casos donde al menos uno de los metodos cometio un error. Se puede observar que en varios casos el modelo de lenguaje fallo, pero la heuristica logro encontrar la correccion correcta; por ejemplo, en thzir → their, el modelo LM selecciono this, mientras que la heuristica selecciono correctamente their. Tambien ocurre algo similar con errores como wold → would, coud → could y borher → brother. Sin embargo, hay otros casos donde ambos metodos fallaron, como shrt → short, mens → means o cred → cried. Para estos errores es necesario revisar si la palabra correcta fue generada dentro del conjunto de candidatos: si no fue generada, el problema esta en la generacion de candidatos; si si estaba disponible pero no fue elegida, entonces el problema esta en la seleccion.